In [0]:
source("utils.R")

is_databricks <- function() {
  nzchar(Sys.getenv("DATABRICKS_RUNTIME_VERSION"))
}

packages <- if (is_databricks()) {
  c("sparklyr", "DBI", "testthat", "arrow")
} else {
  c("DBI", "odbc", "testthat", "arrow")
}

install_if_missing <- function(pkgs) {
  missing <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(missing)) {
    install.packages(missing, dependencies = TRUE)
  }
}

install_if_missing(packages)
lapply(packages, library, character.only = TRUE)

table_name <- "catalog_40_copper_statistics_services.analytics_app.dashboard_last_updated"

connect_databricks <- function(
  catalog = "catalog_40_copper_statistics_services"
) {
  if (is_databricks()) {
    message("Running on Databricks: using sparklyr")

    sc <- sparklyr::spark_connect(method = "databricks")
    return(sc)

  } else {
    message("Running locally: using Databricks ODBC")

    con <- DBI::dbConnect(
      odbc::databricks(),
      driver = "Databricks ODBC Driver",
      workspace = Sys.getenv("DATABRICKS_HOST"),
      httpPath = Sys.getenv("DATABRICKS_CLUSTER_PATH"),
      catalog = catalog,
      useNativeQuery = FALSE
    )
    return(con)
  }
}

In [0]:
create_table_query <- paste("
CREATE TABLE IF NOT EXISTS", table_name, "(
  last_updated TIMESTAMP,
  latest_data DATE
);
")

insert_data_query <- paste("
INSERT OVERWRITE TABLE", table_name, "(last_updated, latest_data)
VALUES (current_timestamp(), current_date() - INTERVAL 2 DAY);
")

dbExecute(connect_databricks, create_table_query)
dbExecute(connect_databricks, insert_data_query)

In [0]:
max_date_query <- function(table) {
  paste0("
    SELECT MAX(date) as max_date
    FROM catalog_40_copper_statistics_services.dashboard_analytics_app.", table)
}

latest_data <- dbGetQuery(connect_databricks, paste("SELECT latest_data FROM", table_name, ""))$latest_data

app_tables <- c("dashboard_publication_summary", "dashboard_service_summary")

# Throw an error (and therefore trigger alert) if any of the dates don't match
for (table in app_tables) {
  max_date <- dbGetQuery(connect_databricks, max_date_query(table))$max_date

  test_that("Max date matches latest_data", {
    expect_equal(max_date, latest_data)
  })
}